# ARCC latest-quarter holdings

This notebook extracts every individual security in Ares Capital's newest 10-Q Schedule of Investments. The first code cell performs the complete extraction and validation. The cells below expose the detailed and company-level pandas DataFrames.

In [1]:
# Standard-library imports used for configuration, parsing, and warnings.
import os
import re
import warnings
from io import StringIO
from pathlib import Path

# Third-party imports used for SEC HTML parsing and tabular data handling.
import pandas as pd
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning

# Keep downloaded EDGAR data inside this project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd()
os.environ.setdefault("EDGAR_LOCAL_DATA_DIR", str(PROJECT_ROOT / ".edgar-data"))

import edgar

# SEC requests require an identifying name and email.
# Read it at runtime so the identity is not stored in the notebook.
sec_identity = os.getenv("EDGAR_IDENTITY")
if not sec_identity:
    sec_identity = input(
        "Enter your SEC identity (for example: Your Name your.email@example.com): "
    )
edgar.set_identity(sec_identity)

# The filing tables use fixed column positions after being read by pandas.
# Keeping these positions in one dictionary makes the row parsing easier to follow.
TICKER = "ARCC"
COLUMN_POSITIONS = {
    "company_as_reported": 0,
    "business_description": 6,
    "investment_type": 12,
    "coupon_pct": 18,
    "reference_rate": 21,
    "spread_pct": 24,
    "acquisition_date": 30,
    "maturity_date": 36,
    "shares_units": 42,
    "principal_usd_millions": 46,
    "amortized_cost_usd_millions": 52,
    "fair_value_usd_millions": 58,
    "footnotes": 60,
}
SCHEDULE_HEADER_MARKERS = (
    "Company (1)",
    "Business Description",
    "Investment (18)",
    "Amortized Cost",
    "Fair Value",
)


# Convert a table value into a consistent, trimmed string or None.
def clean_text(value):
    if pd.isna(value):
        return None

    text = str(value).replace("\ufffd", " ")
    text = re.sub(r"\s+", " ", text).strip()

    if not text or text.lower() == "nan":
        return None
    return text


# Safely read one position from a pandas row.
def cell(row, position):
    if position not in row.index:
        return None
    return clean_text(row[position])


# Convert a formatted accounting or percentage value into a number.
def number(value, left=None, right=None):
    if value is None:
        return None

    text = value.replace(",", "")
    text = text.replace("$", "").replace("%", "").strip()

    if text in {"", "-", "--", "—"}:
        return None

    # Parentheses can mark a negative value. Some filings place them in a
    # neighboring cell, so inspect the cells on either side as well.
    negative = (
        (text.startswith("(") and text.endswith(")"))
        or (left is not None and "(" in left)
        or (right is not None and ")" in right)
    )
    text = text.strip("() ")

    try:
        result = float(text)
    except ValueError:
        return None

    return -result if negative else result


# Accounting values sometimes span three adjacent cells in the source table.
def accounting_cell(row, center):
    value = cell(row, center)
    left_value = cell(row, center - 1)
    right_value = cell(row, center + 1)
    return number(value, left_value, right_value)


# Remove footnote numbers appended to a reported company name.
def clean_company_name(name):
    return re.sub(r"(?:\s*\(\d+\))+$", "", name).strip()


# Identify Schedule of Investments tables by their header text.
def is_schedule_table(text):
    normalized_text = text.lower()
    return all(marker.lower() in normalized_text for marker in SCHEDULE_HEADER_MARKERS)


# Retrieve the newest unamended ARCC 10-Q filing.
arcc = edgar.Company(TICKER)
ten_q_filings = arcc.get_filings(form="10-Q", amendments=False)
latest_10q_filing = ten_q_filings.latest()
report_date = str(ten_q_filings.to_pandas().iloc[0]["reportDate"])

# Parse the filing HTML and locate the current-period Schedule of Investments.
# The filing can contain more than one related table, so collect all tables
# from the first schedule through its Total Investments row.
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
soup = BeautifulSoup(latest_10q_filing.html(), "lxml")
schedule_tables = []
reported_cost = None
reported_fair_value = None
schedule_started = False

for table in soup.find_all("table"):
    table_text = " ".join(table.get_text(" ", strip=True).split())

    if not schedule_started:
        if not is_schedule_table(table_text):
            continue
        schedule_started = True

    if is_schedule_table(table_text):
        schedule_tables.append(table)

    # The first Total Investments row gives us independent totals for validation.
    if schedule_started and "Total Investments" in table_text:
        total_frame = pd.read_html(StringIO(str(table)), flavor="lxml", header=None)[0]
        total_rows = total_frame[
            total_frame[0].astype(str).str.contains(
                "Total Investments", case=False, na=False
            )
        ]
        total_row = total_rows.iloc[-1]

        reported_cost = accounting_cell(
            total_row, COLUMN_POSITIONS["amortized_cost_usd_millions"]
        )
        reported_fair_value = accounting_cell(
            total_row, COLUMN_POSITIONS["fair_value_usd_millions"]
        )
        break

if not schedule_tables or reported_fair_value is None:
    raise RuntimeError("The current Schedule of Investments could not be located.")


# Extract one record for each loan, equity security, or other investment.
records = []
current_sector = None
current_company = None
current_company_reported = None
current_description = None

for schedule_page, table in enumerate(schedule_tables, start=1):
    frame = pd.read_html(StringIO(str(table)), flavor="lxml", header=None)[0]

    for _, row in frame.iterrows():
        company_text = cell(row, COLUMN_POSITIONS["company_as_reported"])
        description = cell(row, COLUMN_POSITIONS["business_description"])
        investment_type = cell(row, COLUMN_POSITIONS["investment_type"])

        # Repeated headers can appear when a table continues onto another page.
        if company_text == "Company (1)" or investment_type == "Investment (18)":
            continue
        if company_text and company_text.lower().startswith("total investments"):
            break

        # A sector heading has text in the company column but no holding data.
        if company_text and not description and not investment_type:
            financial_columns = (
                "shares_units",
                "principal_usd_millions",
                "amortized_cost_usd_millions",
                "fair_value_usd_millions",
            )
            has_financial_data = any(
                cell(row, COLUMN_POSITIONS[key]) is not None
                for key in financial_columns
            )
            if not has_financial_data:
                current_sector = company_text
                continue

        # A blank company cell means this is another investment in the
        # company shown on the preceding row.
        if company_text:
            current_company_reported = company_text
            current_company = clean_company_name(company_text)
            current_description = description
        elif description:
            current_description = description

        if not investment_type:
            continue
        if current_company is None:
            raise RuntimeError(
                f"Investment appeared before a company name on page {schedule_page}."
            )

        record = {
            "company": current_company,
            "company_as_reported": current_company_reported,
            "sector": current_sector,
            "business_description": current_description,
            "investment_type": investment_type,
            "coupon_pct": number(cell(row, COLUMN_POSITIONS["coupon_pct"])),
            "reference_rate": cell(row, COLUMN_POSITIONS["reference_rate"]),
            "spread_pct": number(cell(row, COLUMN_POSITIONS["spread_pct"])),
            "acquisition_date": cell(row, COLUMN_POSITIONS["acquisition_date"]),
            "maturity_date": cell(row, COLUMN_POSITIONS["maturity_date"]),
            "shares_units": number(cell(row, COLUMN_POSITIONS["shares_units"])),
            "principal_usd_millions": accounting_cell(
                row, COLUMN_POSITIONS["principal_usd_millions"]
            ),
            "amortized_cost_usd_millions": accounting_cell(
                row, COLUMN_POSITIONS["amortized_cost_usd_millions"]
            ),
            "fair_value_usd_millions": accounting_cell(
                row, COLUMN_POSITIONS["fair_value_usd_millions"]
            ),
            "footnotes": cell(row, COLUMN_POSITIONS["footnotes"]),
            "schedule_page": schedule_page,
        }
        records.append(record)


# Build the detailed DataFrame with one row per individual security.
holdings = pd.DataFrame.from_records(records)
holdings.insert(0, "holding_id", range(1, len(holdings) + 1))
holdings.insert(1, "ticker", TICKER)
holdings.insert(2, "report_date", report_date)
holdings.insert(3, "filing_date", str(latest_10q_filing.filing_date))
holdings.insert(4, "accession_number", latest_10q_filing.accession_number)

# Group individual securities into a company-level summary.
company_summary = (
    holdings.groupby(
        ["company", "sector", "business_description"],
        dropna=False,
        as_index=False,
    )
    .agg(
        security_count=("holding_id", "count"),
        principal_usd_millions=("principal_usd_millions", "sum"),
        amortized_cost_usd_millions=("amortized_cost_usd_millions", "sum"),
        fair_value_usd_millions=("fair_value_usd_millions", "sum"),
    )
    .sort_values("fair_value_usd_millions", ascending=False)
    .reset_index(drop=True)
)
company_summary.insert(0, "ticker", TICKER)
company_summary.insert(1, "report_date", report_date)
company_summary["portfolio_fair_value_pct"] = (
    company_summary["fair_value_usd_millions"]
    / company_summary["fair_value_usd_millions"].sum()
    * 100
)

# Compare extracted values with the filing totals and the grouped summary.
extracted_cost = holdings["amortized_cost_usd_millions"].sum()
extracted_fair_value = holdings["fair_value_usd_millions"].sum()
summary_fair_value = company_summary["fair_value_usd_millions"].sum()
tolerance = 0.11

validation_results = pd.DataFrame(
    [
        {
            "check": "Every security has a company",
            "observed": int(holdings["company"].notna().sum()),
            "expected": len(holdings),
            "passed": holdings["company"].notna().all(),
        },
        {
            "check": "Every security has an investment type",
            "observed": int(holdings["investment_type"].notna().sum()),
            "expected": len(holdings),
            "passed": holdings["investment_type"].notna().all(),
        },
        {
            "check": "Amortized cost reconciles ($ millions)",
            "observed": round(extracted_cost, 1),
            "expected": reported_cost,
            "passed": abs(extracted_cost - reported_cost) <= tolerance,
        },
        {
            "check": "Fair value reconciles ($ millions)",
            "observed": round(extracted_fair_value, 1),
            "expected": reported_fair_value,
            "passed": abs(extracted_fair_value - reported_fair_value) <= tolerance,
        },
        {
            "check": "Company summary reconciles to detail ($ millions)",
            "observed": round(summary_fair_value, 1),
            "expected": round(extracted_fair_value, 1),
            "passed": abs(summary_fair_value - extracted_fair_value) <= tolerance,
        },
    ]
)

assert validation_results["passed"].all(), validation_results

print(
    f"Loaded {len(holdings):,} securities across "
    f"{holdings['company'].nunique():,} companies for {report_date}."
)
print("Total fair value: $" + f"{extracted_fair_value:,.1f} million")
print("All validation checks passed.")

c:\Users\talan\OneDrive\BDC Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 1,439 securities across 593 companies for 2026-06-30.
Total fair value: $29,349.3 million
All validation checks passed.


## Detailed holdings DataFrame

`holdings` contains one row per individual loan, equity security, or other investment. In VS Code, use the table/data viewer to browse and filter all rows.

In [2]:
pd.set_option("display.max_columns", None)
holdings.tail(10)

,holding_id,ticker,report_date,filing_date,accession_number,company,company_as_reported,sector,business_description,investment_type,coupon_pct,reference_rate,spread_pct,acquisition_date,maturity_date,shares_units,principal_usd_millions,amortized_cost_usd_millions,fair_value_usd_millions,footnotes,schedule_page
1429,1430,ARCC,2026-06-30,2026-07-29,0001628280-26-050307,Zeppelin US Buyer Inc. and Providence Equity P...,Zeppelin US Buyer Inc. and Providence Equity P...,Transportation,Specialty logistics platform for high-stakes p...,First lien senior secured loan,8.48,SOFR (Q),4.75,07/2025,08/2032,NaN,11.0,11.0,10.9,(2)(9),67
1430,1431,ARCC,2026-06-30,2026-07-29,0001628280-26-050307,Zeppelin US Buyer Inc. and Providence Equity P...,Zeppelin US Buyer Inc. and Providence Equity P...,Transportation,Specialty logistics platform for high-stakes p...,Limited partnership interest,NaN,NaN,NaN,07/2025,NaN,1475181.0,NaN,1.5,1.7,(2),67
1431,1432,ARCC,2026-06-30,2026-07-29,0001628280-26-050307,"Ferrellgas, L.P. and Ferrellgas Partners, L.P.","Ferrellgas, L.P. and Ferrellgas Partners, L.P.",Gas Utilities,Distributor of propane and related accessories,Senior preferred units,9.71,NaN,NaN,03/2021,NaN,64155.0,NaN,64.2,73.4,NaN,67
1432,1433,ARCC,2026-06-30,2026-07-29,0001628280-26-050307,"Ferrellgas, L.P. and Ferrellgas Partners, L.P.","Ferrellgas, L.P. and Ferrellgas Partners, L.P.",Gas Utilities,Distributor of propane and related accessories,Class A units,NaN,NaN,NaN,09/2022,NaN,476770.0,NaN,15.4,11.4,(2)(16),67
1433,1434,ARCC,2026-06-30,2026-07-29,0001628280-26-050307,"Opal Fuels Intermediate HoldCo LLC, and Opal F...","Opal Fuels Intermediate HoldCo LLC, and Opal F...",Gas Utilities,Owner of natural gas facilities,First lien senior secured loan,7.23,SOFR (Q),3.50,09/2023,09/2028,NaN,0.1,0.1,0.1,(2)(6),67
1434,1435,ARCC,2026-06-30,2026-07-29,0001628280-26-050307,"Opal Fuels Intermediate HoldCo LLC, and Opal F...","Opal Fuels Intermediate HoldCo LLC, and Opal F...",Gas Utilities,Owner of natural gas facilities,Class A common stock,NaN,NaN,NaN,07/2022,NaN,3059533.0,NaN,23.3,6.7,(6)(16),67
1435,1436,ARCC,2026-06-30,2026-07-29,0001628280-26-050307,"Expereo USA, Inc. and Ristretto Bidco B.V.","Expereo USA, Inc. and Ristretto Bidco B.V. (13)",Telecommunication Services,Global internet managed service provider,First lien senior secured revolving loan,9.69,SOFR (Q),6.00,12/2024,12/2030,NaN,3.5,3.5,3.4,(2)(6)(9),67
1436,1437,ARCC,2026-06-30,2026-07-29,0001628280-26-050307,"Expereo USA, Inc. and Ristretto Bidco B.V.","Expereo USA, Inc. and Ristretto Bidco B.V. (13)",Telecommunication Services,Global internet managed service provider,First lien senior secured loan,NaN,SOFR (Q),6.50,12/2024,12/2030,NaN,4.4,4.4,4.2,(2)(6)(9),67
1437,1438,ARCC,2026-06-30,2026-07-29,0001628280-26-050307,"Expereo USA, Inc. and Ristretto Bidco B.V.","Expereo USA, Inc. and Ristretto Bidco B.V. (13)",Telecommunication Services,Global internet managed service provider,First lien senior secured loan,NaN,SOFR (Q),6.50,12/2024,12/2030,NaN,58.3,58.3,55.9,(2)(6)(9),67
1438,1439,ARCC,2026-06-30,2026-07-29,0001628280-26-050307,Odevo AB,Odevo AB (13),Real Estate Management and Development,Technology-enabled provider of residential pro...,First lien senior secured loan,8.50,SONIA (Q),4.75,09/2025,12/2030,NaN,2.4,2.4,2.4,(2)(6),67


## Company-level summary

`company_summary` combines all securities belonging to the same portfolio company.

In [3]:
company_summary

,ticker,report_date,company,sector,business_description,security_count,principal_usd_millions,amortized_cost_usd_millions,fair_value_usd_millions,portfolio_fair_value_pct
0,ARCC,2026-06-30,"Ivy Hill Asset Management, L.P.",Financial Services,Asset management services,2,946.0,2646.5,2842.8,9.686091
1,ARCC,2026-06-30,"Senior Direct Lending Program, LLC",Investment Funds and Vehicles,Co-investment vehicle,2,1153.7,1140.4,1153.7,3.930929
2,ARCC,2026-06-30,"High Street Buyer, Inc. and High Street Holdco...",Insurance,Insurance brokerage platform,10,45.4,353.9,384.3,1.309401
3,ARCC,2026-06-30,"Denali Intermediate Holdings, Inc. and Denali ...",Commercial and Professional Services,Provider of business decisioning data and anal...,3,384.1,389.9,372.0,1.267492
4,ARCC,2026-06-30,Reddy Ice LLC,Consumer Distribution and Retail,Packaged ice manufacturer and distributor,5,343.6,343.6,343.6,1.170726
...,...,...,...,...,...,...,...,...,...,...
588,ARCC,2026-06-30,"Research Now Group, LLC and Dynata, LLC and Ne...",Commercial and Professional Services,Provider of outsourced data collection to the ...,2,0.0,0.0,0.0,0.000000
589,ARCC,2026-06-30,CREST Exeter Street Solar 2004-1,Investment Funds and Vehicles,Investment vehicle,1,0.0,0.0,0.0,0.000000
590,ARCC,2026-06-30,"CMW Parent LLC (fka Black Arrow, Inc.)","Sports, Media and Entertainment",Multiplatform media firm,1,0.0,0.0,0.0,0.000000
591,ARCC,2026-06-30,"ESCP PPG Holdings, LLC",Capital Goods,Distributor of new equipment and aftermarket p...,2,0.0,5.8,0.0,0.000000


## Validation

These checks prove that required fields are present and that extracted amortized cost and fair value reconcile to the totals printed in the filing.

In [4]:
validation_results

,check,observed,expected,passed
0,Every security has a company,1439.0,1439.0,True
1,Every security has an investment type,1439.0,1439.0,True
2,Amortized cost reconciles ($ millions),29674.6,29674.6,True
3,Fair value reconciles ($ millions),29349.3,29349.3,True
4,Company summary reconciles to detail ($ millions),29349.3,29349.3,True
